In [5]:
!pip install -q pymupdf google-genai numpy
from google.colab import userdata
from google import genai
from google.genai import types
import numpy as np
import fitz

API_KEY = userdata.get("GEMINI_API_KEY")
client = genai.Client(api_key=API_KEY)

EMBEDDING_MODEL = "gemini-embedding-001"
GENERATION_MODEL = "gemini-3.8-flash"

print("Gemini connected successfully!")




Gemini connected successfully!


In [6]:
from google.colab import files

uploaded = files.upload()

pdf_path = next(iter(uploaded.keys()))

doc = fitz.open(pdf_path)

print("PDF uploaded:", pdf_path)
print("Number of pages:", len(doc))

Saving rag_qa_btech_regulations_practice.pdf to rag_qa_btech_regulations_practice (1).pdf
PDF uploaded: rag_qa_btech_regulations_practice (1).pdf
Number of pages: 8


In [8]:
full_text = ""

for page in doc:
    full_text += page.get_text() + "\n"

print("Text extracted successfully!")
print("Total characters:", len(full_text))
print(full_text[:500])

Text extracted successfully!
Total characters: 10690
RAG QA Practice Document
Page 1
B.Tech Academic Regulations
RAG Question-Answering Practice Document
Purpose: This short, searchable-text document is prepared as an input PDF for a Retrieval-Augmented
Generation (RAG) coursework exercise. It summarizes selected topics found in Aditya University's publicly
available B.Tech AR23 academic regulations, such as attendance, promotion, credits, grading, and examinations.
Important: This is a study/practice summary, not a replacement for the complete of


In [9]:
def create_chunks(text, chunk_size=800, overlap=150):
    chunks = []
    start = 0
    chunk_id = 0

    while start < len(text):
        end = start + chunk_size

        chunks.append({
            "chunk_id": chunk_id,
            "text": text[start:end]
        })

        chunk_id += 1
        start += chunk_size - overlap

    return chunks


chunks = create_chunks(
    full_text,
    chunk_size=800,
    overlap=150
)

print("Total chunks:", len(chunks))
print("First chunk:", chunks[0]["text"])

Total chunks: 17
First chunk: RAG QA Practice Document
Page 1
B.Tech Academic Regulations
RAG Question-Answering Practice Document
Purpose: This short, searchable-text document is prepared as an input PDF for a Retrieval-Augmented
Generation (RAG) coursework exercise. It summarizes selected topics found in Aditya University's publicly
available B.Tech AR23 academic regulations, such as attendance, promotion, credits, grading, and examinations.
Important: This is a study/practice summary, not a replacement for the complete official regulations. For formal
decisions, check the official university document.
Suggested RAG task: split the PDF into chunks, embed the chunks, retrieve the top three chunks for each
question, and answer only from the retrieved text.
Document topics
•
Attendance requirements and shortage of atten


In [10]:
def get_embedding(text, task_type):
    result = client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=text,
        config=types.EmbedContentConfig(
            task_type=task_type,
            output_dimensionality=768
        )
    )

    return np.array(
        result.embeddings[0].values,
        dtype=np.float32
    )


for chunk in chunks:
    chunk["embedding"] = get_embedding(
        chunk["text"],
        "RETRIEVAL_DOCUMENT"
    )

print("Embeddings generated:", len(chunks))
print("Embedding dimensions:", len(chunks[0]["embedding"]))

Embeddings generated: 17
Embedding dimensions: 768


In [11]:
def cosine_similarity(vector1, vector2):
    denominator = (
        np.linalg.norm(vector1) *
        np.linalg.norm(vector2)
    )

    if denominator == 0:
        return 0.0

    return float(np.dot(vector1, vector2) / denominator)


def search_chunks(question, chunks, top_k=3):
    question_embedding = get_embedding(
        question,
        "RETRIEVAL_QUERY"
    )

    results = []

    for chunk in chunks:
        score = cosine_similarity(
            question_embedding,
            chunk["embedding"]
        )

        results.append({
            "chunk_id": chunk["chunk_id"],
            "score": score,
            "text": chunk["text"]
        })

    results.sort(
        key=lambda item: item["score"],
        reverse=True
    )

    return results[:top_k]


test_question = (
    "What is the minimum attendance required in each course?"
)

retrieved_chunks = search_chunks(
    test_question,
    chunks,
    top_k=3
)

print("Question:", test_question)

for item in retrieved_chunks:
    print("\nChunk ID:", item["chunk_id"])
    print("Similarity score:", round(item["score"], 4))
    print("Text:", item["text"])

Question: What is the minimum attendance required in each course?

Chunk ID: 12
Similarity score: 0.6999
Text:  case.
Makeup internal examinations
If a student is absent from an internal examination for a valid reason and has prior permission from the Head of
Department, the student may attend a makeup internal examination. The schedule is provided in the academic
calendar.

RAG QA Practice Document
Page 7
6. Retrieval Practice: Questions with Answers
Q1. What minimum attendance is required in each course?
A1. The B.Tech AR23 regulations state that a student needs at least 40% attendance in each course.
Q2. What aggregate attendance is required to appear for Semester End Examinations?
A2. The required aggregate attendance is 75%.
Q3. Can attendance below 65% in aggregate be condoned?
A3. No. The regulations state that aggregate attendance below 65% cannot be condoned.
Q4. How many instructional wee

Chunk ID: 3
Similarity score: 0.699
Text:  student is not eligible to take that semeste

In [14]:
def answer_question(question, chunks):
    retrieved = search_chunks(
        question,
        chunks,
        top_k=3
    )

    context = ""

    for item in retrieved:
        context += (
            f"\n[Chunk ID: {item['chunk_id']}]\n"
            f"{item['text']}\n"
        )

    prompt = f"""
You are a document-based question-answering assistant.

Answer the question using ONLY the retrieved context below.

Rules:
1. Do not use outside knowledge.
2. Do not guess or invent facts.
3. If the context does not contain the answer, say:
   "I don't know. The answer is not found in the document."
4. Keep the answer clear and concise.
5. Mention the chunk ID that supports your answer.

RETRIEVED CONTEXT:
{context}

QUESTION:
{question}

ANSWER:
"""

    response = client.models.generate_content(
        model=GENERATION_MODEL,
        contents=prompt
    )

    print("\nQuestion:", question)
    print("\nAnswer:")
    print(response.text)

    print("\nRetrieved source chunks:")

    for item in retrieved:
        print(
            f"Chunk ID: {item['chunk_id']} | "
            f"Similarity: {item['score']:.4f}"
        )

    print("-" * 60)

    return response.text

In [15]:
answerable_questions = [
    "What is the minimum attendance required in each course?",
    "What aggregate attendance is required for Semester End Examinations?",
    "How many instructional weeks are in a semester?"
]

for question in answerable_questions:
    answer_question(question, chunks)


Question: What is the minimum attendance required in each course?

Answer:
Based on the provided context, the minimum attendance required in each course is at least 40% [Chunk ID: 2, Chunk ID: 3, Chunk ID: 12].

Retrieved source chunks:
Chunk ID: 12 | Similarity: 0.6999
Chunk ID: 3 | Similarity: 0.6990
Chunk ID: 2 | Similarity: 0.6898
------------------------------------------------------------

Question: What aggregate attendance is required for Semester End Examinations?

Answer:
A student must acquire at least 75% attendance in aggregate across all courses to be eligible for the Semester End Examinations. 

(Supported by [Chunk ID: 2], [Chunk ID: 3], and [Chunk ID: 12])

Retrieved source chunks:
Chunk ID: 2 | Similarity: 0.7922
Chunk ID: 3 | Similarity: 0.7787
Chunk ID: 12 | Similarity: 0.7685
------------------------------------------------------------

Question: How many instructional weeks are in a semester?

Answer:
A semester consists of 16 instructional weeks. [Chunk ID: 6] (

In [16]:
unanswerable_questions = [
    "What is the exact tuition fee for the next academic year?",
    "What is the date of the next university holiday?",
    "Who obtained the highest marks in the latest semester?"
]

for question in unanswerable_questions:
    answer_question(question, chunks)


Question: What is the exact tuition fee for the next academic year?

Answer:
I don't know. The answer is not found in the document. [Chunk ID: 14, Chunk ID: 15]

Retrieved source chunks:
Chunk ID: 14 | Similarity: 0.6376
Chunk ID: 15 | Similarity: 0.6075
Chunk ID: 10 | Similarity: 0.5795
------------------------------------------------------------

Question: What is the date of the next university holiday?

Answer:
I don't know. The answer is not found in the document. [Chunk ID: 15]

Retrieved source chunks:
Chunk ID: 14 | Similarity: 0.6396
Chunk ID: 15 | Similarity: 0.6304
Chunk ID: 13 | Similarity: 0.5887
------------------------------------------------------------

Question: Who obtained the highest marks in the latest semester?

Answer:
I don't know. The answer is not found in the document. [Chunk ID: 14, 15]

Retrieved source chunks:
Chunk ID: 15 | Similarity: 0.7181
Chunk ID: 14 | Similarity: 0.6504
Chunk ID: 10 | Similarity: 0.6361
--------------------------------------------

In [17]:
large_chunks = create_chunks(
    full_text,
    chunk_size=3000,
    overlap=150
)

print("Number of chunks with size 3000:", len(large_chunks))

for chunk in large_chunks:
    chunk["embedding"] = get_embedding(
        chunk["text"],
        "RETRIEVAL_DOCUMENT"
    )

print("Large-chunk embeddings generated.")

Number of chunks with size 3000: 4
Large-chunk embeddings generated.


In [19]:
from google.genai.errors import ClientError

print("RESULTS USING CHUNK SIZE 3000")

for question in answerable_questions:
    try:
        answer_question(question, large_chunks)
    except ClientError as e:
        if "429" in str(e) or "RESOURCE_EXHAUSTED" in str(e):
            print("\nGemini API quota exceeded.")
            print("Wait for quota to reset, then rerun this question:")
            print(question)
            break
        else:
            raise

RESULTS USING CHUNK SIZE 3000

Question: What is the minimum attendance required in each course?

Answer:
Under the B.Tech AR23 regulations, a student must acquire at least 40% attendance in each course.

**Supporting Chunk ID:** Chunk ID: 0 (also mentioned in Chunk ID: 2)

Retrieved source chunks:
Chunk ID: 0 | Similarity: 0.6905
Chunk ID: 2 | Similarity: 0.6723
Chunk ID: 3 | Similarity: 0.6712
------------------------------------------------------------

Question: What aggregate attendance is required for Semester End Examinations?

Answer:
The required aggregate attendance to be eligible for the Semester End Examinations is 75% across all courses. 

(Supported by [Chunk ID: 0] and [Chunk ID: 2])

Retrieved source chunks:
Chunk ID: 0 | Similarity: 0.7290
Chunk ID: 2 | Similarity: 0.7238
Chunk ID: 3 | Similarity: 0.7065
------------------------------------------------------------

Question: How many instructional weeks are in a semester?

Answer:
A semester consists of 16 instructiona